# Mmvlm4SCD on Google Colab — **real Nigeria cohort** (NDHS 2018)

End-to-end training uses **released microdata** from the Nigeria **Demographic and Health Survey 2018** Household Recode Stata dataset (**NGHR7BDT**), which contains **real** sickle genotyping (RDT result **`sb113b`**: AA / AS / AC / SC / SS / Other), **child hemoglobin** (**`hc53`**), anthropometry (**`hc1`–`hc3`**), and child sex (**`hc27`** when present).

Clinical and genomic inputs are grounded in those fields. **Imaging** and **longitudinal vital** modalities are **not** collected in NDHS → they stay **zero tensors** here (explicit limitation). Survival columns are censored placeholders; training sets **`beta=0`** so the Cox loss is disabled.

Register (free) with [The DHS Program](https://dhsprogram.com/data/new-user-registration.cfm), download **NGHR7BDT** (Stata) for Nigeria Standard DHS 2018 from the dataset listing, unzip, and upload **`NGHR7BDT.dta`** on Colab (or mount Google Drive).

**Citation:** NPC Nigeria and ICF (2019). *Nigeria Demographic and Health Survey 2018.*
Variable dictionary: **`sb113b`** “Result of genotype RDT”; see [World Bank NG microdata catalog](https://microdata.worldbank.org/index.php/catalog/3540) / DHS RECODE docs.

## 1. Environment setup (Colab or local Jupyter)

- **Colab:** set ``MMVLM_REPO_URL`` to your fork (then run the setup cell).
- Obtain **NGHR7BDT.dta** after DHS login; set ``NIGERIA_DHS_NGHR_DTA=/path/to/NGHR7BDT.dta`` or upload when prompted in the next section.

In [ ]:
import os
import subprocess
import sys

def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

def _find_repo_root(start: str) -> str:
    cur = os.path.abspath(start)
    for _ in range(6):
        if os.path.isdir(os.path.join(cur, "src", "mmvlm4scd")):
            return cur
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    raise RuntimeError("Could not find mmvlm4scd package root (missing src/mmvlm4scd)")

if _in_colab():
    REPO_URL = os.environ.get(
        "MMVLM_REPO_URL",
        "https://github.com/yourusername/Mmvlm4SCD.git",
    )
    DEST = "/content/Mmvlm4SCD"
    if not os.path.isdir(os.path.join(DEST, "src", "mmvlm4scd")):
        subprocess.check_call(
            ["git", "clone", "--depth", "1", REPO_URL, DEST],
            stdout=subprocess.DEVNULL,
        )
    os.chdir(DEST)
    ROOT = DEST
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
else:
    ROOT = _find_repo_root(os.getcwd())
    os.chdir(ROOT)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])

sys.path.insert(0, os.path.join(ROOT, "src"))
print("Repo root:", ROOT)

## 2. Load **real** Nigeria NDHS 2018 rows (Household Recode)

Resolves ``NIGERIA_DHS_NGHR_DTA`` from the environment, or **Colab file upload**. If neither is provided, raises with instructions (no synthetic fallback).

Optional: enforce ``PIP_INDEX_URL`` / offline installs if your runtime blocks PyPI — this section only uses ``pandas``.

In [ ]:
import os
from pathlib import Path

from mmvlm4scd.data import build_cohort_from_nigeria_dhs2018_hr


def resolve_nigeria_dhs_stata_path() -> Path:
    env = os.environ.get("NIGERIA_DHS_NGHR_DTA")
    if env:
        p = Path(env).expanduser()
        if p.is_file():
            return p
        raise FileNotFoundError(f"NIGERIA_DHS_NGHR_DTA points to missing path: {p}")
    cwd = Path("NGHR7BDT.dta").resolve()
    if cwd.is_file():
        return cwd
    if _in_colab():
        from google.colab import files as colab_files  # noqa: WPS433

        print("Upload NGHR7BDT.dta from NGHR7BDT.zip (DHS Nigeria 2018 Household Recode)…")
        up = colab_files.upload()
        names = list(up.keys())
        if len(names) != 1:
            raise RuntimeError("Please upload exactly one .dta file")
        wrote = Path("/content") / names[0]
        wrote.write_bytes(up[names[0]])
        return wrote
    raise FileNotFoundError(
        "Set NIGERIA_DHS_NGHR_DTA to NGHR7BDT.dta or copy that file into the repo root."
    )


TIMESTEPS = 24

dta_path = resolve_nigeria_dhs_stata_path()
print("Stata:", dta_path)

cohort = build_cohort_from_nigeria_dhs2018_hr(
    dta_path,
    timesteps=TIMESTEPS,
    max_patients=None,
)

print(cohort["meta"])
print("Patients:", len(cohort["clinical"]))
cohort["clinical"].head()




## 3. Train and evaluate on real NDHS modalities

Mirrors ``run_full_experiment.py`` but **does not synthesize subjects**. Imaging and temporal modalities are zeros; **survival placeholders** are meaningless here — **`beta=0`** turns off Cox loss (severity cross-entropy only). Tune ``epochs`` / ``batch_size`` for Colab GPU/CPU limits.

In [ ]:
import os
from pathlib import Path

import numpy as np
import torch

from mmvlm4scd.data import StandardPreprocessor
from mmvlm4scd.data.dataloaders import make_loaders
from mmvlm4scd.data.synthetic import split_indices
from mmvlm4scd.evaluation import evaluate_model_full
from mmvlm4scd.models import ModelConfig, MultimodalSCDModel
from mmvlm4scd.training import Trainer, TrainConfig
from mmvlm4scd.utils import auto_device, set_seed


_cfg_seed = int(os.environ.get("MMVLM_SEED", "7"))

cfg = {
    "model": {
        "embed_dim": 64,
        "fusion": "attention",
        "dropout": 0.1,
        "num_severity_classes": 3,
    },
    "train": {
        "epochs": 12,
        "batch_size": 64,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "grad_clip": 1.0,
        "alpha": 1.0,
        # NDHS cohort: no real survival outcome here — Cox term disabled.
        "beta": 0.0,
        "early_stop_patience": 6,
        "select_metric": "auroc_ovr",
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "seed": _cfg_seed,
    },
}

set_seed(cfg["train"]["seed"])

pre = StandardPreprocessor().fit(cohort["clinical"])
clin_x = pre.transform(cohort["clinical"])
tr_idx, va_idx, te_idx = split_indices(
    len(cohort["severity"]), seed=_cfg_seed
)

_bs = cfg["train"]["batch_size"]
if len(tr_idx) < _bs * 3:
    _bs = max(8, len(tr_idx) // 8)
    cfg["train"]["batch_size"] = _bs
    print(f"Adjusted batch_size to {_bs} (small train split)")

tr_loader, va_loader, te_loader = make_loaders(
    cohort,
    clin_x,
    tr_idx,
    va_idx,
    te_idx,
    batch_size=cfg["train"]["batch_size"],
)

device = cfg["train"].get("device") or auto_device()
model = MultimodalSCDModel(
    ModelConfig(
        clinical_input_dim=clin_x.shape[1],
        genomic_input_dim=cohort["genomic"].shape[1],
        imaging_input_dim=cohort["imaging"].shape[1],
        temporal_input_dim=cohort["temporal"].shape[2],
        embed_dim=cfg["model"]["embed_dim"],
        fusion=cfg["model"]["fusion"],
        dropout=cfg["model"]["dropout"],
        num_severity_classes=cfg["model"]["num_severity_classes"],
    )
)

trainer = Trainer(
    model,
    TrainConfig(
        epochs=cfg["train"]["epochs"],
        lr=cfg["train"]["lr"],
        weight_decay=cfg["train"]["weight_decay"],
        grad_clip=cfg["train"]["grad_clip"],
        alpha=cfg["train"]["alpha"],
        beta=cfg["train"]["beta"],
        early_stop_patience=cfg["train"]["early_stop_patience"],
        select_metric=cfg["train"]["select_metric"],
        device=device,
    ),
)

train_out = trainer.fit(tr_loader, va_loader)
print("Best val score:", train_out["best_score"])

ev = evaluate_model_full(model, te_loader, device=device)
metrics = ev["metrics"]
for k in sorted(metrics.keys()):
    v = metrics[k]
    if isinstance(v, (float, int, np.floating, np.integer)):
        print(f"{k}: {float(v):.4f}")

out_dir = Path(ROOT) / "experiments" / "results" / "colab_nigeria_ndhs2018"
out_dir.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), out_dir / "last_model.pt")
print("Saved checkpoint to", out_dir / "last_model.pt")

